In [0]:
-- MVP: Construção de Pipeline de Dados na Nuvem
-- Tema: Análise de corridas de táxi de Nova York com Databricks
-- Aluna: Marcela Pholyana de Oliveira da Silva
-- Plataforma: Databricks Free Edition
-- Dataset: samples.nyctaxi.trips

SELECT
  'Notebook iniciado com sucesso' AS status;

status
Notebook iniciado com sucesso


In [0]:
-- Criação de um schema específico para organizar as tabelas do MVP
-- Este schema será usado para armazenar as camadas Bronze, Silver e Gold do pipeline

CREATE SCHEMA IF NOT EXISTS mvp_nyctaxi;

In [0]:
-- Definindo o schema do projeto como padrão para as próximas consultas

USE mvp_nyctaxi;

In [0]:
-- Camada Bronze
-- Esta tabela armazena os dados brutos das corridas de táxi a partir da base pública do Databricks.
-- Nesta etapa, os dados ainda não são tratados; apenas são persistidos no schema do MVP.

CREATE OR REPLACE TABLE bronze_trips AS
SELECT
  tpep_pickup_datetime,
  tpep_dropoff_datetime,
  trip_distance,
  fare_amount,
  pickup_zip,
  dropoff_zip
FROM samples.nyctaxi.trips;

num_affected_rows,num_inserted_rows


In [0]:
-- Conferência da tabela Bronze

SELECT
  COUNT(*) AS total_linhas
FROM bronze_trips;

total_linhas
21932


In [0]:
-- Camada Silver
-- Esta tabela contém os dados tratados das corridas de táxi.
-- Foram removidos registros inconsistentes para as análises:
-- 1. corridas com distância menor ou igual a zero;
-- 2. corridas com tarifa menor ou igual a zero;
-- 3. corridas com horário de fim menor ou igual ao horário de início.
-- Também foram criados campos derivados para apoiar as análises.

CREATE OR REPLACE TABLE silver_trips AS
SELECT
  tpep_pickup_datetime,
  tpep_dropoff_datetime,
  DATE(tpep_pickup_datetime) AS pickup_date,
  HOUR(tpep_pickup_datetime) AS pickup_hour,
  trip_distance,
  fare_amount,
  pickup_zip,
  dropoff_zip,
  ROUND(
    (UNIX_TIMESTAMP(tpep_dropoff_datetime) - UNIX_TIMESTAMP(tpep_pickup_datetime)) / 60,
    2
  ) AS trip_duration_minutes
FROM bronze_trips
WHERE trip_distance > 0
  AND fare_amount > 0
  AND tpep_dropoff_datetime > tpep_pickup_datetime;

num_affected_rows,num_inserted_rows


In [0]:
-- Conferência da camada Silver
-- Compara a quantidade de linhas da camada Bronze com a camada Silver.

SELECT
  (SELECT COUNT(*) FROM bronze_trips) AS linhas_bronze,
  (SELECT COUNT(*) FROM silver_trips) AS linhas_silver,
  (SELECT COUNT(*) FROM bronze_trips) - (SELECT COUNT(*) FROM silver_trips) AS linhas_removidas;

linhas_bronze,linhas_silver,linhas_removidas
21932,21847,85


In [0]:
-- Tabela de resumo da qualidade dos dados
-- Esta tabela registra os principais problemas encontrados na camada Bronze.

CREATE OR REPLACE TABLE gold_quality_summary AS
SELECT
  COUNT(*) AS total_linhas_bronze,

  SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS qtd_distancia_nula,
  SUM(CASE WHEN fare_amount IS NULL THEN 1 ELSE 0 END) AS qtd_tarifa_nula,
  SUM(CASE WHEN pickup_zip IS NULL THEN 1 ELSE 0 END) AS qtd_pickup_zip_nulo,
  SUM(CASE WHEN dropoff_zip IS NULL THEN 1 ELSE 0 END) AS qtd_dropoff_zip_nulo,

  SUM(CASE WHEN trip_distance <= 0 THEN 1 ELSE 0 END) AS qtd_distancia_zero_ou_negativa,
  SUM(CASE WHEN fare_amount <= 0 THEN 1 ELSE 0 END) AS qtd_tarifa_zero_ou_negativa,
  SUM(CASE WHEN tpep_dropoff_datetime <= tpep_pickup_datetime THEN 1 ELSE 0 END) AS qtd_tempo_invalido,

  (SELECT COUNT(*) FROM bronze_trips) - (SELECT COUNT(*) FROM silver_trips) AS qtd_linhas_removidas_na_silver

FROM bronze_trips;

num_affected_rows,num_inserted_rows


In [0]:
-- Visualização do resumo de qualidade dos dados

SELECT *
FROM gold_quality_summary;

total_linhas_bronze,qtd_distancia_nula,qtd_tarifa_nula,qtd_pickup_zip_nulo,qtd_dropoff_zip_nulo,qtd_distancia_zero_ou_negativa,qtd_tarifa_zero_ou_negativa,qtd_tempo_invalido,qtd_linhas_removidas_na_silver
21932,0,0,0,0,76,10,1,85


In [0]:
-- Camada Gold
-- Esta tabela agrega as corridas por dia, criando indicadores prontos para análise.
-- Ela será usada para responder perguntas de negócio sobre volume, tarifa, distância e duração das corridas.

CREATE OR REPLACE TABLE gold_daily_metrics AS
SELECT
  pickup_date,
  COUNT(*) AS total_corridas,
  ROUND(SUM(fare_amount), 2) AS receita_total,
  ROUND(AVG(fare_amount), 2) AS tarifa_media,
  ROUND(AVG(trip_distance), 2) AS distancia_media,
  ROUND(AVG(trip_duration_minutes), 2) AS duracao_media_minutos,
  ROUND(SUM(trip_distance), 2) AS distancia_total
FROM silver_trips
GROUP BY pickup_date;

num_affected_rows,num_inserted_rows


In [0]:
-- Visualização dos indicadores diários

SELECT *
FROM gold_daily_metrics
ORDER BY pickup_date;

pickup_date,total_corridas,receita_total,tarifa_media,distancia_media,duracao_media_minutos,distancia_total
2016-01-01,326,3998.0,12.26,3.13,20.14,1021.86
2016-01-02,337,4022.0,11.93,3.01,28.24,1012.9
2016-01-03,275,3403.0,12.37,3.19,11.06,876.85
2016-01-04,322,3890.5,12.08,3.0,11.56,967.3
2016-01-05,357,4054.0,11.36,2.61,19.9,933.42
2016-01-06,358,4133.0,11.54,2.63,12.4,942.92
2016-01-07,353,4294.0,12.16,2.94,16.97,1036.59
2016-01-08,390,4934.0,12.65,2.94,17.25,1145.75
2016-01-09,410,4858.0,11.85,2.85,15.75,1168.69
2016-01-10,345,4050.5,11.74,3.03,15.1,1046.01


In [0]:
-- Camada Gold
-- Esta tabela agrega as corridas por CEP de origem.
-- Ela permite analisar quais regiões de origem concentram maior volume de corridas e receita.

CREATE OR REPLACE TABLE gold_pickup_zip_metrics AS
SELECT
  pickup_zip,
  COUNT(*) AS total_corridas,
  ROUND(SUM(fare_amount), 2) AS receita_total,
  ROUND(AVG(fare_amount), 2) AS tarifa_media,
  ROUND(AVG(trip_distance), 2) AS distancia_media,
  ROUND(AVG(trip_duration_minutes), 2) AS duracao_media_minutos
FROM silver_trips
GROUP BY pickup_zip;

num_affected_rows,num_inserted_rows


In [0]:
-- Top 10 CEPs de origem por quantidade de corridas

SELECT *
FROM gold_pickup_zip_metrics
ORDER BY total_corridas DESC
LIMIT 10;

pickup_zip,total_corridas,receita_total,tarifa_media,distancia_media,duracao_media_minutos
10001,1227,13028.01,10.62,2.21,14.51
10003,1180,12963.0,10.99,2.33,12.45
10011,1128,12319.0,10.92,2.29,14.99
10021,1017,10321.5,10.15,2.03,11.53
10018,1010,11539.01,11.42,2.58,15.62
10023,1006,10112.0,10.05,2.12,12.39
10028,927,9432.5,10.18,2.21,12.31
10012,831,9456.0,11.38,2.44,13.25
10110,761,8261.5,10.86,2.31,12.25
10065,700,6837.5,9.77,1.97,16.21


In [0]:
-- Análise 1: resumo geral do período analisado
-- Esta consulta responde ao volume total de corridas, receita total,
-- tarifa média, distância média e duração média das corridas após o tratamento dos dados.

SELECT
  COUNT(*) AS total_corridas_validas,
  ROUND(SUM(fare_amount), 2) AS receita_total,
  ROUND(AVG(fare_amount), 2) AS tarifa_media,
  ROUND(AVG(trip_distance), 2) AS distancia_media,
  ROUND(AVG(trip_duration_minutes), 2) AS duracao_media_minutos,
  MIN(pickup_date) AS data_inicial,
  MAX(pickup_date) AS data_final
FROM silver_trips;

total_corridas_validas,receita_total,tarifa_media,distancia_media,duracao_media_minutos,data_inicial,data_final
21847,269022.03,12.31,2.86,15.16,2016-01-01,2016-02-29


In [0]:
-- Análise 2: dias com maior volume de corridas
-- Esta consulta identifica os dias com maior quantidade de corridas válidas.

SELECT
  pickup_date,
  total_corridas,
  receita_total,
  tarifa_media,
  distancia_media,
  duracao_media_minutos
FROM gold_daily_metrics
ORDER BY total_corridas DESC
LIMIT 10;

pickup_date,total_corridas,receita_total,tarifa_media,distancia_media,duracao_media_minutos
2016-02-11,456,5913.0,12.97,2.79,17.59
2016-01-30,451,5239.5,11.62,2.6,16.05
2016-02-26,450,5823.5,12.94,2.96,17.68
2016-01-21,440,5558.51,12.63,2.78,17.5
2016-02-13,438,4923.0,11.24,2.44,15.79
2016-02-06,436,5002.0,11.47,2.59,12.23
2016-02-12,428,6071.0,14.18,3.26,21.74
2016-02-18,420,5037.5,11.99,2.68,23.35
2016-01-22,417,4672.5,11.21,2.4,12.78
2016-01-16,415,5171.5,12.46,3.01,16.2


In [0]:
-- Análise 3: dias com maior receita
-- Esta consulta identifica os dias com maior soma de tarifas.

SELECT
  pickup_date,
  total_corridas,
  receita_total,
  tarifa_media,
  distancia_media,
  duracao_media_minutos
FROM gold_daily_metrics
ORDER BY receita_total DESC
LIMIT 10;

pickup_date,total_corridas,receita_total,tarifa_media,distancia_media,duracao_media_minutos
2016-02-12,428,6071.0,14.18,3.26,21.74
2016-02-11,456,5913.0,12.97,2.79,17.59
2016-02-26,450,5823.5,12.94,2.96,17.68
2016-01-21,440,5558.51,12.63,2.78,17.5
2016-02-21,408,5446.0,13.35,3.47,14.21
2016-02-24,403,5346.5,13.27,2.98,14.55
2016-01-14,393,5307.0,13.5,3.28,17.68
2016-02-04,392,5243.5,13.38,3.05,15.29
2016-01-30,451,5239.5,11.62,2.6,16.05
2016-01-16,415,5171.5,12.46,3.01,16.2


In [0]:
-- Análise 4: CEPs de origem com maior receita
-- Esta consulta identifica os CEPs de origem que mais geraram receita total.

SELECT
  pickup_zip,
  total_corridas,
  receita_total,
  tarifa_media,
  distancia_media,
  duracao_media_minutos
FROM gold_pickup_zip_metrics
ORDER BY receita_total DESC
LIMIT 10;

pickup_zip,total_corridas,receita_total,tarifa_media,distancia_media,duracao_media_minutos
11422,421,19041.0,45.23,15.8,42.61
11371,482,14762.5,30.63,9.67,30.62
10001,1227,13028.01,10.62,2.21,14.51
10003,1180,12963.0,10.99,2.33,12.45
10011,1128,12319.0,10.92,2.29,14.99
10018,1010,11539.01,11.42,2.58,15.62
10021,1017,10321.5,10.15,2.03,11.53
10023,1006,10112.0,10.05,2.12,12.39
10012,831,9456.0,11.38,2.44,13.25
10028,927,9432.5,10.18,2.21,12.31


In [0]:
-- Análise 5: resumo da qualidade dos dados
-- Esta consulta consolida os problemas encontrados na camada Bronze
-- e mostra o impacto do tratamento realizado na camada Silver.

SELECT *
FROM gold_quality_summary;

total_linhas_bronze,qtd_distancia_nula,qtd_tarifa_nula,qtd_pickup_zip_nulo,qtd_dropoff_zip_nulo,qtd_distancia_zero_ou_negativa,qtd_tarifa_zero_ou_negativa,qtd_tempo_invalido,qtd_linhas_removidas_na_silver
21932,0,0,0,0,76,10,1,85


In [0]:
-- Conferência das tabelas criadas no schema do MVP
-- Esta consulta evidencia que as tabelas Bronze, Silver e Gold foram persistidas na plataforma de nuvem.

SHOW TABLES IN mvp_nyctaxi;

database,tableName,isTemporary
mvp_nyctaxi,bronze_trips,false
mvp_nyctaxi,gold_daily_metrics,false
mvp_nyctaxi,gold_pickup_zip_metrics,false
mvp_nyctaxi,gold_quality_summary,false
mvp_nyctaxi,silver_trips,false


In [0]:
-- PRINT 1: Evidência das tabelas criadas e persistidas no Databricks

SHOW TABLES IN mvp_nyctaxi;

database,tableName,isTemporary
mvp_nyctaxi,bronze_trips,false
mvp_nyctaxi,gold_daily_metrics,false
mvp_nyctaxi,gold_pickup_zip_metrics,false
mvp_nyctaxi,gold_quality_summary,false
mvp_nyctaxi,silver_trips,false


In [0]:
-- PRINT 2: Evidência da análise de qualidade dos dados

SELECT *
FROM gold_quality_summary;

total_linhas_bronze,qtd_distancia_nula,qtd_tarifa_nula,qtd_pickup_zip_nulo,qtd_dropoff_zip_nulo,qtd_distancia_zero_ou_negativa,qtd_tarifa_zero_ou_negativa,qtd_tempo_invalido,qtd_linhas_removidas_na_silver
21932,0,0,0,0,76,10,1,85


In [0]:
-- PRINT 3: Resumo geral do período analisado

SELECT
  COUNT(*) AS total_corridas_validas,
  ROUND(SUM(fare_amount), 2) AS receita_total,
  ROUND(AVG(fare_amount), 2) AS tarifa_media,
  ROUND(AVG(trip_distance), 2) AS distancia_media,
  ROUND(AVG(trip_duration_minutes), 2) AS duracao_media_minutos,
  MIN(pickup_date) AS data_inicial,
  MAX(pickup_date) AS data_final
FROM silver_trips;

total_corridas_validas,receita_total,tarifa_media,distancia_media,duracao_media_minutos,data_inicial,data_final
21847,269022.03,12.31,2.86,15.16,2016-01-01,2016-02-29


In [0]:
-- PRINT 4: Top 10 dias com maior volume de corridas

SELECT
  pickup_date,
  total_corridas,
  receita_total,
  tarifa_media,
  distancia_media,
  duracao_media_minutos
FROM gold_daily_metrics
ORDER BY total_corridas DESC
LIMIT 10;

pickup_date,total_corridas,receita_total,tarifa_media,distancia_media,duracao_media_minutos
2016-02-11,456,5913.0,12.97,2.79,17.59
2016-01-30,451,5239.5,11.62,2.6,16.05
2016-02-26,450,5823.5,12.94,2.96,17.68
2016-01-21,440,5558.51,12.63,2.78,17.5
2016-02-13,438,4923.0,11.24,2.44,15.79
2016-02-06,436,5002.0,11.47,2.59,12.23
2016-02-12,428,6071.0,14.18,3.26,21.74
2016-02-18,420,5037.5,11.99,2.68,23.35
2016-01-22,417,4672.5,11.21,2.4,12.78
2016-01-16,415,5171.5,12.46,3.01,16.2


In [0]:
-- PRINT 5: Top 10 CEPs de origem com maior receita

SELECT
  pickup_zip,
  total_corridas,
  receita_total,
  tarifa_media,
  distancia_media,
  duracao_media_minutos
FROM gold_pickup_zip_metrics
ORDER BY receita_total DESC
LIMIT 10;

pickup_zip,total_corridas,receita_total,tarifa_media,distancia_media,duracao_media_minutos
11422,421,19041.0,45.23,15.8,42.61
11371,482,14762.5,30.63,9.67,30.62
10001,1227,13028.01,10.62,2.21,14.51
10003,1180,12963.0,10.99,2.33,12.45
10011,1128,12319.0,10.92,2.29,14.99
10018,1010,11539.01,11.42,2.58,15.62
10021,1017,10321.5,10.15,2.03,11.53
10023,1006,10112.0,10.05,2.12,12.39
10012,831,9456.0,11.38,2.44,13.25
10028,927,9432.5,10.18,2.21,12.31


1. Exportar ou baixar o notebook/código do Databricks
2. Criar repositório no GitHub
3. Subir notebook + prints
4. Montar README.md com explicação do MVP